# Data Preparation

In [1]:
import pandas as pd
df = pd.read_csv("C:\\Users\\kusha\\OneDrive\\Desktop\\DA\\My projects\\Marketing Campaign Analysis\\raw dataset\\global_ads_performance_dataset.csv")

df.head()

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39


In [2]:
#checking data dimension 
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 1800
Columns: 14


In [4]:
#checking whether all date values can be interpreted as valid date 
pd.to_datetime(df['date'], errors = 'coerce').isna().sum()

np.int64(0)

In [7]:
#Comvereting date column into datetime
df['date'] = pd.to_datetime(df['date'])

In [8]:
df['date'].dtype

dtype('<M8[ns]')

## Validating CTR calculation

In [12]:
calculated_ctr = df['clicks'] / df['impressions']
print("Maximum CTR difference:", (df['CTR']- calculated_ctr).abs().max())

Maximum CTR difference: 0.00017707840640349137


## Validating CPC calculation

In [13]:
calculated_cpc = df['ad_spend'] / df['clicks']

print(
    "Maximum CPC difference:",
    (df['CPC'] - calculated_cpc).abs().max()
)

Maximum CPC difference: 4.440892098500626e-16


## Validating CPA calculation

In [14]:
calculated_cpa = df['ad_spend'] / df['conversions']

print(
    "Maximum CPA difference:",
    (df['CPA'] - calculated_cpa).abs().max()
)

Maximum CPA difference: 0.005000000000002558


## Validating ROAS calculation

In [15]:
calculated_roas = df['revenue'] / df['ad_spend']

print(
    "Maximum ROAS difference:",
    (df['ROAS'] - calculated_roas).abs().max()
)

Maximum ROAS difference: 0.004994570272586429


## Checking whether clicks exceed impression

In [16]:
(df['clicks'] > df['impressions']).sum()

np.int64(0)

## Checking whether conversions exceed clicks

In [17]:
(df['conversions'] > df['clicks']).sum()

np.int64(0)

## Checking for zero or negative ad spend

In [18]:
(df['ad_spend'] <= 0).sum()

np.int64(0)

## Checking for negative conversions

In [19]:
(df['conversions'] < 0).sum()

np.int64(0)

## Checking for negative impressions

In [20]:
(df['impressions'] < 0).sum()

np.int64(0)

## Checking for negative clicks

In [21]:
(df['clicks'] < 0).sum()

np.int64(0)

## Checking for negative revenue

In [22]:
(df['revenue'] < 0).sum()

np.int64(0)

## Checking for invalid CTR values

In [23]:
((df['CTR'] < 0) | (df['CTR'] > 1)).sum()

np.int64(0)

In [ ]:
## Check for negative CPC values

In [24]:
(df['CPC'] < 0).sum()

np.int64(0)

## Checking for negative CPA values

In [25]:
(df['CPA'] < 0).sum()

np.int64(0)

## Checking for negative ROAS values

In [26]:
(df['ROAS'] < 0).sum()

np.int64(0)

# Outlier Assessment

In [27]:
# Check for potential outliers in impressions using IQR
Q1 = df['impressions'].quantile(0.25)
Q3 = df['impressions'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['impressions'] < lower_bound) |
    (df['impressions'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers))

Q1: 54948.0
Q3: 150470.25
IQR: 95522.25
Lower bound: -88335.375
Upper bound: 293753.625
Number of potential outliers: 0


In [28]:
# Check for potential outliers in clicks using IQR

Q1 = df['clicks'].quantile(0.25)
Q3 = df['clicks'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['clicks'] < lower_bound) |
    (df['clicks'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers))

Q1: 1678.0
Q3: 5628.0
IQR: 3950.0
Lower bound: -4247.0
Upper bound: 11553.0
Number of potential outliers: 42


In [29]:
# Inspecting potential click outliers

outliers[['date', 'platform', 'campaign_type',
          'industry', 'country', 'impressions',
          'clicks', 'CTR', 'ad_spend',
          'conversions', 'revenue']].sort_values(
              'clicks',
              ascending=False
          ).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,ad_spend,conversions,revenue
863,2024-04-04,TikTok Ads,Search,SaaS,UK,199291,16660,0.0836,19325.60,1016,207611.26
1437,2024-12-22,TikTok Ads,Search,EdTech,Canada,194117,16422,0.0846,21841.26,1133,121361.12
1449,2024-07-06,TikTok Ads,Video,SaaS,Canada,187411,15667,0.0836,6736.81,996,51799.95
662,2024-02-01,TikTok Ads,Video,Healthcare,Australia,184504,15479,0.0839,12383.20,538,112251.74
751,2024-11-11,TikTok Ads,Shopping,EdTech,India,184969,15407,0.0833,20645.38,352,52669.94
1752,2024-02-02,TikTok Ads,Shopping,Healthcare,Australia,196605,14922,0.0759,26262.72,757,181823.28
115,2024-12-13,TikTok Ads,Shopping,Healthcare,Australia,191994,14898,0.0776,16089.84,936,60355.75
1143,2024-07-21,TikTok Ads,Display,SaaS,UAE,179039,14663,0.0819,19941.68,1151,281693.26
376,2024-09-14,TikTok Ads,Search,E-commerce,Canada,198085,14460,0.0730,11857.20,1133,295028.26
866,2024-03-07,TikTok Ads,Display,Healthcare,Australia,186782,14270,0.0764,15411.60,931,224929.42


In [30]:
# Checking for potential outliers in CTR using IQR

Q1 = df['CTR'].quantile(0.25)
Q3 = df['CTR'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_ctr = df[
    (df['CTR'] < lower_bound) |
    (df['CTR'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_ctr))

Q1: 0.0254
Q3: 0.0498
IQR: 0.024399999999999998
Lower bound: -0.011199999999999995
Upper bound: 0.08639999999999999
Number of potential outliers: 12


In [31]:
# Inspecting potential CTR outliers

outliers_ctr[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'ad_spend',
     'conversions', 'revenue', 'ROAS']
].sort_values(
    'CTR',
    ascending=False
)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,ad_spend,conversions,revenue,ROAS
1582,2024-03-22,TikTok Ads,Shopping,Fintech,UK,40308,3853,0.0956,4199.77,291,33059.93,7.87
17,2024-08-20,TikTok Ads,Search,Fintech,Germany,38828,3607,0.0929,3498.79,116,8788.51,2.51
1687,2024-08-11,TikTok Ads,Search,Fintech,Germany,101411,9370,0.0924,7589.70,523,26633.21,3.51
876,2024-04-06,TikTok Ads,Video,EdTech,Germany,60438,5554,0.0919,7997.76,286,35506.60,4.44
239,2024-12-16,TikTok Ads,Search,Fintech,Canada,33302,3007,0.0903,841.96,95,3253.81,3.86
1783,2024-04-12,TikTok Ads,Shopping,SaaS,UAE,152322,13678,0.0898,5607.98,813,103237.84,18.41
1038,2024-04-14,TikTok Ads,Video,Healthcare,USA,118687,10586,0.0892,7939.50,635,88046.75,11.09
746,2024-07-14,TikTok Ads,Display,E-commerce,Canada,113621,10112,0.0890,10010.88,472,84899.73,8.48
845,2024-01-01,TikTok Ads,Display,SaaS,USA,103026,9066,0.0880,12964.38,278,46631.00,3.60
1384,2024-01-28,TikTok Ads,Display,Healthcare,India,104626,9144,0.0874,10058.40,389,83739.39,8.33


In [32]:
# Checking for potential outliers in CPC using IQR

Q1 = df['CPC'].quantile(0.25)
Q3 = df['CPC'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_cpc = df[
    (df['CPC'] < lower_bound) |
    (df['CPC'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_cpc))

Q1: 0.95
Q3: 2.05
IQR: 1.0999999999999999
Lower bound: -0.7
Upper bound: 3.6999999999999997
Number of potential outliers: 7


In [33]:
# Inspecting potential CPC outliers

outliers_cpc[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'CPC', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'CPC',
    ascending=False
)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
605,2024-04-14,Google Ads,Shopping,E-commerce,India,9791,519,0.0531,3.95,2050.05,40,51.25,2255.24,1.10
440,2024-05-02,Google Ads,Video,EdTech,UAE,117333,2898,0.0247,3.93,11389.14,48,237.27,5367.73,0.47
911,2024-09-07,Google Ads,Shopping,EdTech,USA,91150,4119,0.0452,3.83,15775.77,98,160.98,4547.95,0.29
1070,2024-08-19,Google Ads,Shopping,Healthcare,Germany,125019,8251,0.0660,3.77,31106.27,609,51.08,135443.82,4.35
895,2024-11-17,Google Ads,Search,EdTech,UK,115282,4876,0.0423,3.76,18333.76,77,238.10,2427.15,0.13
1350,2024-05-11,Google Ads,Search,SaaS,Canada,89833,4536,0.0505,3.74,16964.64,166,102.20,42497.44,2.51
1209,2024-12-29,Google Ads,Shopping,SaaS,Canada,149438,3287,0.0220,3.73,12260.51,153,80.13,8351.95,0.68


In [34]:
# Check for potential outliers in ad spend using IQR

Q1 = df['ad_spend'].quantile(0.25)
Q3 = df['ad_spend'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_ad_spend = df[
    (df['ad_spend'] < lower_bound) |
    (df['ad_spend'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_ad_spend))

Q1: 1966.5875
Q3: 8455.83
IQR: 6489.2425
Lower bound: -7767.276250000001
Upper bound: 18189.69375
Number of potential outliers: 90


In [35]:
# Inspect potential ad spend outliers

outliers_ad_spend[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'CPC', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'ad_spend',
    ascending=False
).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
711,2024-09-12,Google Ads,Video,E-commerce,UK,193869,13124,0.0677,2.93,38453.32,264,145.66,67604.89,1.76
945,2024-11-29,Google Ads,Search,Healthcare,UK,195606,10817,0.0553,3.53,38184.01,122,312.98,7355.91,0.19
24,2024-04-09,Google Ads,Display,Healthcare,Australia,198824,9046,0.0455,3.61,32656.06,407,80.24,71482.19,2.19
77,2024-04-08,Google Ads,Video,E-commerce,USA,198555,13104,0.0660,2.46,32235.84,336,95.94,58794.32,1.82
1070,2024-08-19,Google Ads,Shopping,Healthcare,Germany,125019,8251,0.0660,3.77,31106.27,609,51.08,135443.82,4.35
624,2024-07-21,Google Ads,Video,SaaS,Canada,157890,9204,0.0583,3.37,31017.48,321,96.63,10260.33,0.33
1788,2024-06-01,Google Ads,Shopping,SaaS,UAE,154088,9106,0.0591,3.24,29503.44,294,100.35,42565.78,1.44
1354,2024-10-29,Google Ads,Shopping,EdTech,USA,182395,8517,0.0467,3.45,29383.65,517,56.83,27133.78,0.92
1525,2024-04-14,Google Ads,Search,E-commerce,Australia,191372,11711,0.0612,2.48,29043.28,335,86.70,75028.42,2.58
1322,2024-03-08,Google Ads,Search,SaaS,USA,185923,7827,0.0421,3.68,28803.36,215,133.97,61106.18,2.12


In [36]:
# Checking for potential outliers in conversions using IQR

Q1 = df['conversions'].quantile(0.25)
Q3 = df['conversions'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_conversions = df[
    (df['conversions'] < lower_bound) |
    (df['conversions'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_conversions))

Q1: 59.0
Q3: 252.25
IQR: 193.25
Lower bound: -230.875
Upper bound: 542.125
Number of potential outliers: 85


In [37]:
# Inspecting potential conversion outliers

outliers_conversions[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'conversions',
    ascending=False
).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,ad_spend,conversions,CPA,revenue,ROAS
1143,2024-07-21,TikTok Ads,Display,SaaS,UAE,179039,14663,0.0819,19941.68,1151,17.33,281693.26,14.13
376,2024-09-14,TikTok Ads,Search,E-commerce,Canada,198085,14460,0.0730,11857.20,1133,10.47,295028.26,24.88
1437,2024-12-22,TikTok Ads,Search,EdTech,Canada,194117,16422,0.0846,21841.26,1133,19.28,121361.12,5.56
863,2024-04-04,TikTok Ads,Search,SaaS,UK,199291,16660,0.0836,19325.60,1016,19.02,207611.26,10.74
1449,2024-07-06,TikTok Ads,Video,SaaS,Canada,187411,15667,0.0836,6736.81,996,6.76,51799.95,7.69
772,2024-12-12,TikTok Ads,Search,Healthcare,Canada,163984,12971,0.0791,21791.28,962,22.65,170813.74,7.84
1096,2024-11-07,TikTok Ads,Display,Healthcare,UK,187199,12036,0.0643,16007.88,946,16.92,165825.37,10.36
115,2024-12-13,TikTok Ads,Shopping,Healthcare,Australia,191994,14898,0.0776,16089.84,936,17.19,60355.75,3.75
984,2024-07-22,Google Ads,Video,Healthcare,USA,190826,12174,0.0638,21304.50,933,22.83,139478.01,6.55
866,2024-03-07,TikTok Ads,Display,Healthcare,Australia,186782,14270,0.0764,15411.60,931,16.55,224929.42,14.59


In [38]:
# Checking for potential outliers in CPA using IQR

Q1 = df['CPA'].quantile(0.25)
Q3 = df['CPA'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_cpa = df[
    (df['CPA'] < lower_bound) |
    (df['CPA'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_cpa))

Q1: 20.2025
Q3: 56.8125
IQR: 36.61
Lower bound: -34.7125
Upper bound: 111.72749999999999
Number of potential outliers: 140


In [39]:
# Inspect potential CPA outliers

outliers_cpa[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'CPC', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'CPA',
    ascending=False
).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
1466,2024-06-27,Google Ads,Display,EdTech,UAE,36391,1684,0.0463,3.59,6045.56,18,335.86,4868.51,0.81
945,2024-11-29,Google Ads,Search,Healthcare,UK,195606,10817,0.0553,3.53,38184.01,122,312.98,7355.91,0.19
921,2024-12-30,Google Ads,Shopping,Healthcare,Canada,148041,8053,0.0544,3.47,27943.91,98,285.14,4146.07,0.15
1711,2024-08-04,Google Ads,Display,Healthcare,Canada,144494,8409,0.0582,3.04,25563.36,93,274.87,20233.22,0.79
66,2024-05-13,Google Ads,Video,E-commerce,Australia,34402,1314,0.0382,3.29,4323.06,17,254.30,1273.24,0.29
133,2024-01-10,Google Ads,Video,SaaS,UK,170268,7917,0.0465,3.14,24859.38,98,253.67,22672.12,0.91
225,2024-11-08,Google Ads,Shopping,Fintech,India,32518,1258,0.0387,2.41,3031.78,12,252.65,2688.72,0.89
757,2024-11-28,Google Ads,Video,Healthcare,Germany,24755,811,0.0328,2.48,2011.28,8,251.41,1786.84,0.89
1488,2024-11-23,Google Ads,Display,Healthcare,Canada,192822,6228,0.0323,3.35,20863.80,87,239.81,18207.05,0.87
895,2024-11-17,Google Ads,Search,EdTech,UK,115282,4876,0.0423,3.76,18333.76,77,238.10,2427.15,0.13


In [40]:
# Checking for potential outliers in revenue using IQR

Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_revenue = df[
    (df['revenue'] < lower_bound) |
    (df['revenue'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_revenue))

Q1: 7275.7575
Q3: 38963.385
IQR: 31687.627500000002
Lower bound: -40255.683750000004
Upper bound: 86494.82625000001
Number of potential outliers: 127


In [41]:
# Inspecting potential revenue outliers

outliers_revenue[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'revenue',
    ascending=False
).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,ad_spend,conversions,CPA,revenue,ROAS
376,2024-09-14,TikTok Ads,Search,E-commerce,Canada,198085,14460,0.0730,11857.20,1133,10.47,295028.26,24.88
1143,2024-07-21,TikTok Ads,Display,SaaS,UAE,179039,14663,0.0819,19941.68,1151,17.33,281693.26,14.13
1132,2024-03-10,TikTok Ads,Search,E-commerce,Australia,179818,11724,0.0652,12075.72,880,13.72,242039.99,20.04
866,2024-03-07,TikTok Ads,Display,Healthcare,Australia,186782,14270,0.0764,15411.60,931,16.55,224929.42,14.59
291,2024-04-13,Google Ads,Video,EdTech,Australia,192135,10663,0.0555,18980.14,741,25.61,213823.36,11.27
297,2024-05-19,TikTok Ads,Shopping,SaaS,India,176946,13784,0.0779,11854.24,743,15.95,208986.19,17.63
250,2024-09-21,TikTok Ads,Display,Healthcare,UK,135956,11529,0.0848,5764.50,732,7.88,207824.45,36.05
863,2024-04-04,TikTok Ads,Search,SaaS,UK,199291,16660,0.0836,19325.60,1016,19.02,207611.26,10.74
1795,2024-07-11,TikTok Ads,Video,E-commerce,Germany,180128,10555,0.0586,8866.20,707,12.54,200566.90,22.62
1388,2024-03-06,Google Ads,Shopping,SaaS,UAE,193326,12334,0.0638,15787.52,715,22.08,196162.94,12.43


In [42]:
# Checking for potential outliers in ROAS using IQR

Q1 = df['ROAS'].quantile(0.25)
Q3 = df['ROAS'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_roas = df[
    (df['ROAS'] < lower_bound) |
    (df['ROAS'] > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers_roas))

Q1: 2.17
Q3: 8.2125
IQR: 6.0425
Lower bound: -6.893750000000001
Upper bound: 17.27625
Number of potential outliers: 123


In [43]:
# Inspecting potential ROAS outliers

outliers_roas[
    ['date', 'platform', 'campaign_type',
     'industry', 'country', 'impressions',
     'clicks', 'CTR', 'CPC', 'ad_spend',
     'conversions', 'CPA', 'revenue', 'ROAS']
].sort_values(
    'ROAS',
    ascending=False
).head(10)

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
485,2024-05-07,TikTok Ads,Display,SaaS,UK,145497,4743,0.0326,0.35,1660.05,313,5.30,81349.35,49.00
1528,2024-12-29,TikTok Ads,Display,E-commerce,Germany,66086,3317,0.0502,0.33,1094.61,206,5.31,51119.73,46.70
29,2024-05-23,TikTok Ads,Shopping,EdTech,Canada,91900,4696,0.0511,0.53,2488.88,367,6.78,109696.83,44.07
835,2024-12-18,TikTok Ads,Shopping,Fintech,USA,169238,6583,0.0389,0.53,3488.99,507,6.88,146453.92,41.98
212,2024-11-29,TikTok Ads,Search,E-commerce,Germany,159692,6882,0.0431,0.37,2546.34,523,4.87,106309.31,41.75
889,2024-10-03,TikTok Ads,Search,Healthcare,Australia,141087,10609,0.0752,0.44,4667.96,737,6.33,190244.62,40.76
1691,2024-12-08,TikTok Ads,Video,Fintech,Germany,116184,4089,0.0352,0.34,1390.26,263,5.29,54249.41,39.02
1369,2024-09-15,Meta Ads,Shopping,Healthcare,UK,174862,3846,0.0220,0.45,1730.70,299,5.79,67036.78,38.73
1551,2024-03-20,TikTok Ads,Video,SaaS,USA,191380,8171,0.0427,0.50,4085.50,572,7.14,155778.92,38.13
1015,2024-02-01,TikTok Ads,Shopping,Fintech,India,95288,7108,0.0746,0.51,3625.08,454,7.98,132401.61,36.52


## Data Standardization check

In [44]:
# Checking  for leading/trailing whitespace in categorical columns

categorical_columns = [
    'platform',
    'campaign_type',
    'industry',
    'country'
]

for col in categorical_columns:
    whitespace_count = (
        df[col].astype(str) != df[col].astype(str).str.strip()
    ).sum()
    
    print(f"{col}: {whitespace_count}")

platform: 0
campaign_type: 0
industry: 0
country: 0


In [45]:
# Check capitalization patterns in categorical columns

for col in categorical_columns:
    print(f"\n{col}:")
    print(df[col].value_counts())


platform:
platform
Google Ads    720
Meta Ads      630
TikTok Ads    450
Name: count, dtype: int64

campaign_type:
campaign_type
Search      477
Video       456
Shopping    447
Display     420
Name: count, dtype: int64

industry:
industry
EdTech        372
SaaS          370
Fintech       361
E-commerce    349
Healthcare    348
Name: count, dtype: int64

country:
country
UK           266
USA          266
Canada       262
India        261
UAE          258
Germany      255
Australia    232
Name: count, dtype: int64


In [46]:
# Checking for non-printable characters in categorical columns

for col in categorical_columns:
    invalid_count = df[col].apply(
        lambda x: not str(x).isprintable()
    ).sum()
    
    print(f"{col}: {invalid_count}")

platform: 0
campaign_type: 0
industry: 0
country: 0


In [47]:
# Checking for missing dates in the observed date range

expected_dates = pd.date_range(
    start=df['date'].min(),
    end=df['date'].max(),
    freq='D'
)

missing_dates = expected_dates.difference(df['date'].unique())

print("Expected dates:", len(expected_dates))
print("Observed dates:", df['date'].nunique())
print("Missing dates:", len(missing_dates))

Expected dates: 365
Observed dates: 360
Missing dates: 5


In [48]:
# Identify missing dates
missing_dates

DatetimeIndex(['2024-04-23', '2024-07-01', '2024-07-15', '2024-08-25',
               '2024-11-18'],
              dtype='datetime64[ns]', freq=None)

In [49]:
print("Minimum date:", df['date'].min())
print("Maximum date:", df['date'].max())
print("Future dates:", (df['date'] > pd.Timestamp.today()).sum())

Minimum date: 2024-01-01 00:00:00
Maximum date: 2024-12-30 00:00:00
Future dates: 0


In [52]:
# Final validation of the prepared dataset
'''This confirms that  preparation steps did not accidentally
change the number of records or columns and that the date
conversion was applied correctly.'''

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Date data type:", df['date'].dtype)

# Confirm the expected dataset structure
assert df.shape == (1800, 14), "Unexpected dataset shape"

# Confirm that the date column is stored as datetime
assert pd.api.types.is_datetime64_any_dtype(df['date']), \
    "Date column is not in datetime format"

print("\nFinal validation passed.")

Rows: 1800
Columns: 14
Date data type: datetime64[ns]

Final validation passed.


## Exporting the prepared dataset

In [53]:
# Export the prepared dataset to a new CSV file.
# This prepared file will be used for the SQL analysis stage.

output_file = "global_ads_performance_prepared.csv"
df.to_csv(output_file, index=False)
print(f"Prepared dataset saved as: {output_file}")

Prepared dataset saved as: global_ads_performance_prepared.csv
